# Glosario Interactivo — Conceptos Introductorios de Feature Engineering

_Referencia rapida de los fundamentos estadisticos y matematicos del Modulo 1._

**Modulo 1 — Feature Engineering & Feature Stores** | DSRP Machine Learning Engineering

---

> 📖 Este notebook es un **glosario de referencia**. Cada concepto incluye una definicion concisa
> y un bloque de codigo ejecutable que lo ilustra.  
> Usalo como consulta cuando encuentres un termino desconocido en los notebooks 1-4.

### Indice

1. [Distribucion sesgada y transformacion log1p](#c1)
2. [Correlacion de Pearson vs Informacion Mutua](#c2)
3. [IQR, cuartiles y boxplot](#c3)
4. [z-score y z-score modificado (MAD)](#c4)
5. [Escalado: StandardScaler, RobustScaler, MinMaxScaler](#c5)
6. [Distancia euclidiana y la necesidad de escalar](#c6)
7. [Variable nominal vs ordinal — decision de encoding](#c7)
8. [Fuga de datos (data leakage) — la regla de oro](#c8)
9. [Pipeline y el paradigma fit / transform](#c9)
10. [PCA — varianza explicada y componentes principales](#c10)
11. [Metricas de regresion: RMSE, MAE, R2, RMSLE](#c11)
12. [Feature Store y sesgo entrenamiento-serving](#c12)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import train_test_split

np.random.seed(42)
plt.rcParams.update({"figure.dpi": 100,
                     "axes.spines.top": False,
                     "axes.spines.right": False})

## 1. Distribucion sesgada y transformacion log1p <a id="c1"></a>

**Definicion.** Una distribucion es **sesgada a la derecha** (positivamente) cuando la cola derecha
es mas larga que la izquierda: hay muchos valores pequenos y pocos extremadamente grandes.
El sesgo (*skewness*) se mide como:

$$\text{skewness} = \frac{1}{n}\sum_i\left(\frac{x_i - \mu}{\sigma}\right)^3$$

- Simetrico: $\approx 0$ | Sesgo derecho: $> 0$ | Sesgo izquierdo: $< 0$

**Problema.** Los modelos lineales y los basados en distancias asumen distribuciones
simetricas. Un sesgo alto distorsiona sus predicciones.

**Solucion: log1p.** La transformacion $x' = \log(1 + x)$ comprime las colas y simetriza
la distribucion. Se usa `log1p` (en lugar de `log`) porque maneja $x = 0$ sin error.

> **Clave para Ames Housing:** `SalePrice` tiene skewness $\approx 1.88$.  
> Tras `log1p` baja a $\approx 0.12$. La competencia de Kaggle evalua con **RMSE sobre
> log(SalePrice)** justamente por esto.

| Distribucion | Skewness | Transformacion recomendada |
|---|---|---|
| Precio, Area, Ingresos | $> 1$ | `log1p` o Box-Cox |
| Ligero sesgo | $0.5$-$1$ | `sqrt(x)` o Yeo-Johnson |
| Simetrica | $\approx 0$ | Ninguna necesaria |
| Sesgo negativo | $< -0.5$ | `x**2` o reflexion + log |

In [ ]:
rng = np.random.default_rng(0)
price = rng.lognormal(mean=12, sigma=0.4, size=1000)

def skewness(a):
    a = np.asarray(a)
    return float(((a - a.mean()) ** 3).mean() / a.std() ** 3)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(price, bins=40, color="#e45756", alpha=0.85)
axes[0].set_title(f"SalePrice original  skew = {skewness(price):+.2f}")
axes[0].set_xlabel("precio")

price_log = np.log1p(price)
axes[1].hist(price_log, bins=40, color="#4c78a8", alpha=0.85)
axes[1].set_title(f"log1p(SalePrice)   skew = {skewness(price_log):+.2f}")
axes[1].set_xlabel("log1p(precio)")

plt.suptitle("Efecto de log1p: de cola derecha a distribucion simetrica",
             fontweight="bold")
plt.tight_layout()
plt.show()
print(f"La inversa de log1p es expm1: expm1(log1p(x)) = x")
print(f"Ejemplo: log1p(100) = {np.log1p(100):.3f}, expm1({np.log1p(100):.3f}) = {np.expm1(np.log1p(100)):.1f}")

## 2. Correlacion de Pearson vs Informacion Mutua <a id="c2"></a>

Ambas miden la relacion entre una feature y el target, pero capturan cosas distintas.

### Correlacion de Pearson $r$

$$r = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_i(x_i-\bar{x})^2 \sum_i(y_i-\bar{y})^2}}, \qquad r \in [-1, 1]$$

- $r = 1$: relacion lineal perfecta positiva | $r = -1$: perfecta negativa | $r = 0$: sin relacion **lineal**
- **Limitacion:** detecta solo dependencia lineal. Si la relacion es cuadratica o ciclica, $r \approx 0$ aunque la feature sea muy informativa.

### Informacion Mutua $I(X; Y)$

$$I(X; Y) = \sum_{x,y} p(x,y) \log \frac{p(x,y)}{p(x)\,p(y)} \geq 0$$

- Mide **cuanta informacion comparte** $X$ con $Y$ independientemente de la forma de la relacion.
- $I = 0$: variables independientes | Mayor $I$ = mas informativa.
- **Captura no linealidad** que Pearson no ve.

| Metrica | Rango | Tipo de relacion | Uso en sklearn |
|---|---|---|---|
| Pearson $r$ | $[-1,1]$ | Solo lineal | `f_regression` |
| Informacion Mutua | $[0, \infty)$ | Cualquier dependencia | `mutual_info_regression` |

In [ ]:
n = 300
x = rng.uniform(-3, 3, n)
y_lineal = 2 * x + rng.normal(0, 0.5, n)
y_cuad   = x ** 2 + rng.normal(0, 0.3, n)
y_indep  = rng.normal(0, 1, n)

casos = [(y_lineal, "Lineal: y=2x+e"), (y_cuad, "Cuadratica: y=x^2+e"), (y_indep, "Independiente")]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, (y, titulo) in zip(axes, casos):
    pearson = float(np.corrcoef(x, y)[0, 1])
    mi = float(mutual_info_regression(
        x.reshape(-1, 1), y, random_state=0)[0])
    ax.scatter(x, y, s=10, alpha=0.4, color="#4c78a8")
    ax.set_title(f"{titulo}\nPearson r={pearson:.2f}  MI={mi:.2f}")
    ax.set_xlabel("x")
axes[0].set_ylabel("y")
plt.suptitle("Pearson capta solo linealidad; MI capta cualquier dependencia",
             fontweight="bold")
plt.tight_layout()
plt.show()

## 3. IQR, cuartiles y boxplot <a id="c3"></a>

**Cuartiles.** Dividen los datos ordenados en cuatro partes iguales:
- $Q_1$ = percentil 25 (25% de los datos por debajo)
- $Q_2$ = mediana (percentil 50)
- $Q_3$ = percentil 75

**Rango Intercuartil (IQR):**
$$\text{IQR} = Q_3 - Q_1$$

Representa el 50% central de los datos. Es una medida **robusta** de dispersion (no le afectan los outliers).

**Vallas de Tukey (regla del boxplot):**
$$\text{Valla inferior} = Q_1 - 1.5 \times \text{IQR}, \qquad \text{Valla superior} = Q_3 + 1.5 \times \text{IQR}$$

Los puntos fuera de estas vallas se marcan como **outliers candidatos**.

| Estadistico | Formula | Robusto a outliers | Uso principal |
|---|---|---|---|
| Media | $\bar{x} = \frac{1}{n}\sum x_i$ | No | Escalado estandar |
| Mediana ($Q_2$) | valor central | Si | Imputacion, escalado robusto |
| IQR | $Q_3 - Q_1$ | Si | Deteccion de outliers, RobustScaler |

In [ ]:
gr_liv = np.concatenate([
    rng.normal(1500, 300, 200),
    np.array([4500, 5000, 4800])
])

q1, q2, q3 = np.percentile(gr_liv, [25, 50, 75])
iqr = q3 - q1
valla_inf = q1 - 1.5 * iqr
valla_sup = q3 + 1.5 * iqr
outliers = gr_liv[(gr_liv < valla_inf) | (gr_liv > valla_sup)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.boxplot(gr_liv, vert=True, patch_artist=True,
            boxprops=dict(facecolor="#cfe8ff"),
            medianprops=dict(color="#e45756", lw=2))
ax1.axhline(q1, ls="--", color="gray", lw=1, label=f"Q1={q1:.0f}")
ax1.axhline(q3, ls="--", color="gray", lw=1, label=f"Q3={q3:.0f}")
ax1.axhline(valla_sup, ls=":", color="#e45756", lw=1.5, label=f"Valla sup={valla_sup:.0f}")
ax1.set_title("Boxplot con vallas de Tukey")
ax1.set_ylabel("GrLivArea")
ax1.legend(fontsize=9)

ax2.hist(gr_liv, bins=25, color="#4c78a8", alpha=0.75)
for o in outliers:
    ax2.axvline(o, color="#e45756", lw=1.5, alpha=0.8)
ax2.axvline(valla_sup, ls="--", color="#e45756", lw=2, label=f"Valla Tukey={valla_sup:.0f}")
ax2.set_title(f"Histograma — IQR={iqr:.0f}")
ax2.set_xlabel("GrLivArea"); ax2.legend(fontsize=9)

plt.suptitle("IQR y vallas de Tukey para detectar outliers", fontweight="bold")
plt.tight_layout(); plt.show()
print(f"Q1={q1:.0f}  Q2={q2:.0f}  Q3={q3:.0f}  IQR={iqr:.0f}")
print(f"Valla superior (Tukey): {valla_sup:.0f}")
print(f"Outliers encontrados: {outliers.round(0)}")

## 4. z-score y z-score modificado (MAD) <a id="c4"></a>

### z-score (estandar)

$$z_i = \frac{x_i - \mu}{\sigma}$$

Un punto es outlier si $|z_i| > 3$ (regla empirica: el 99.7% de una normal cae en $\pm 3\sigma$).

**Problema:** $\mu$ y $\sigma$ son sensibles a outliers. Si hay valores extremos, se inflan
$\mu$ y $\sigma$, lo que hace que el z-score de los outliers parezca mas pequeno (paradoja del
enmascaramiento o *masking*).

### z-score modificado (MAD)

Reemplaza $\mu$ por la **mediana** y $\sigma$ por la **desviacion absoluta mediana** (MAD):

$$\text{MAD} = \text{mediana}(|x_i - \text{mediana}(x)|)$$

$$M_i = \frac{0.6745 \cdot (x_i - \text{mediana}(x))}{\text{MAD}}$$

El factor $0.6745$ hace que, para una normal, $\text{MAD} \approx \sigma$. Outlier si $|M_i| > 3.5$.

| Metodo | Centro | Escala | Robusto | Umbral tipico |
|---|---|---|---|---|
| z-score | Media $\mu$ | Desv. estandar $\sigma$ | No | $|z| > 3$ |
| z-score modificado | Mediana | MAD | Si | $|M| > 3.5$ |

In [ ]:
datos = np.concatenate([rng.normal(1500, 200, 100), [5000, 4800, 5200]])

mu, sigma = datos.mean(), datos.std()
z = (datos - mu) / sigma

mediana = np.median(datos)
mad = np.median(np.abs(datos - mediana))
m_mod = 0.6745 * (datos - mediana) / (mad if mad > 0 else 1e-10)

outliers_z   = datos[np.abs(z) > 3]
outliers_mad = datos[np.abs(m_mod) > 3.5]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, scores, umbral, etiqueta, color in [
        (axes[0], z, 3.0, "z-score (media/std)", "#4c78a8"),
        (axes[1], m_mod, 3.5, "z-score modificado (MAD)", "#54a24b")]:
    normal = np.abs(scores) <= umbral
    ax.scatter(np.where(normal), scores[normal], s=15,
               color=color, alpha=0.6, label="normal")
    ax.scatter(np.where(~normal), scores[~normal], s=60,
               color="#e45756", zorder=5, label="outlier")
    ax.axhline(umbral, ls="--", color="gray", lw=1)
    ax.axhline(-umbral, ls="--", color="gray", lw=1)
    ax.set_title(etiqueta)
    ax.set_xlabel("indice"); ax.set_ylabel("score")
    ax.legend(fontsize=9)

plt.suptitle("z-score vs MAD: el MAD no se deja enganar por los outliers",
             fontweight="bold")
plt.tight_layout(); plt.show()
print(f"Outliers z-score   (|z|>3)  : {outliers_z.round(0)}")
print(f"Outliers MAD       (|M|>3.5): {outliers_mad.round(0)}")

## 5. Escalado: StandardScaler, RobustScaler, MinMaxScaler <a id="c5"></a>

El **escalado** pone todas las features en una escala comparable. Muchos algoritmos
(SVM, k-NN, PCA, regresion lineal con regularizacion) miden distancias o gradientes
y son muy sensibles a la escala.

| Scaler | Transformacion | Resultado | Sensible a outliers | Cuándo usar |
|---|---|---|---|---|
| **StandardScaler** | $z = (x-\mu)/\sigma$ | media=0, std=1 | Si | Mayoria de algoritmos, PCA |
| **RobustScaler** | $(x-\text{med})/(Q_3-Q_1)$ | centrado en mediana | No | Features con outliers |
| **MinMaxScaler** | $(x-x_{\min})/(x_{\max}-x_{\min})$ | rango $[0,1]$ | Mucho | Entradas acotadas (imagenes, NN) |

> **Regla:** ajusta los escaladores **solo en el set de entrenamiento** (`fit` en train,
> `transform` en train y test). Nunca ajustar sobre el conjunto completo (leakage).

In [ ]:
gr_area = np.concatenate([rng.lognormal(7.3, 0.25, 200), [4500, 5000]])
X = gr_area.reshape(-1, 1)

z_std  = StandardScaler().fit_transform(X).ravel()
z_rob  = RobustScaler().fit_transform(X).ravel()
z_mm   = MinMaxScaler().fit_transform(X).ravel()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, data, titulo, color in [
    (axes[0], z_std, "StandardScaler\n(media=0, std=1)", "#4c78a8"),
    (axes[1], z_rob, "RobustScaler\n(mediana=0, IQR)",   "#54a24b"),
    (axes[2], z_mm,  "MinMaxScaler\n(rango [0,1])",      "#f58518"),
]:
    ax.hist(data, bins=25, color=color, alpha=0.8, edgecolor="white")
    ax.axvline(np.median(data), color="red", lw=2, ls="--", label="mediana")
    ax.set_title(titulo)
    ax.set_xlabel("valor escalado")
    ax.legend(fontsize=9)

plt.suptitle("Tres escaladores sobre GrLivArea (con outliers artificiales)",
             fontweight="bold")
plt.tight_layout(); plt.show()

for nombre, data in [("StandardScaler", z_std), ("RobustScaler", z_rob), ("MinMaxScaler", z_mm)]:
    print(f"{nombre:20s}  min={data.min():.2f}  max={data.max():.2f}  "
          f"media={data.mean():.2f}  std={data.std():.2f}")

## 6. Distancia euclidiana y la necesidad de escalar <a id="c6"></a>

**Distancia euclidiana** entre dos puntos $\mathbf{a}, \mathbf{b} \in \mathbb{R}^d$:

$$d(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_{j=1}^{d}(a_j - b_j)^2}$$

**El problema:** si $a_1$ esta en miles (p.ej. metros cuadrados) y $a_2$ en unidades del 1-10
(calidad), la primera dimension **domina** la distancia solo por sus unidades, aunque
en informacion real las dos sean igual de importantes.

**La solucion:** estandarizar antes de calcular cualquier distancia.

Algoritmos que requieren escalar porque usan distancias o gradientes:
- **k-NN, DBSCAN, KNNImputer** (distancia directa)
- **Isolation Forest** (aunque usa arboles, la profundidad de aislamiento puede sesgarse)
- **PCA** (maximiza varianza → domina la feature de mayor escala)
- **Regresion lineal/Ridge con regularizacion** (la penalizacion L1/L2 afecta igual a todos los coef)
- **Redes neuronales** (el gradiente depende de la escala de cada feature)

In [ ]:
X_raw = np.array([[1500, 7], [1800, 8], [1600, 5], [3000, 6], [1550, 9]])
scaler = StandardScaler().fit(X_raw)
X_scaled = scaler.transform(X_raw)

punto = np.array([[1600, 6]])
punto_scaled = scaler.transform(punto)

dist_raw    = np.sqrt(((X_raw - punto)**2).sum(axis=1))
dist_scaled = np.sqrt(((X_scaled - punto_scaled)**2).sum(axis=1))

df_demo = pd.DataFrame({
    "GrLivArea": X_raw[:, 0],
    "OverallQual": X_raw[:, 1],
    "dist_sin_escalar": dist_raw.round(1),
    "dist_escalada": dist_scaled.round(3),
    "vecino_sin_esc": (dist_raw == dist_raw.min()),
    "vecino_esc": (dist_scaled == dist_scaled.min()),
})
print(f"Punto de consulta: GrLivArea=1600, OverallQual=6")
print()
print(df_demo.to_string(index=False))
print("\n-> Sin escalar: 'GrLivArea' domina y elige un vecino diferente al correcto")
print("-> Con escalar: ambas features contribuyen por igual a la distancia")

## 7. Variable nominal vs ordinal — decision de encoding <a id="c7"></a>

| Tipo | Definicion | Ejemplos en Ames | Encoding correcto |
|---|---|---|---|
| **Nominal** | Categorias sin orden natural | Neighborhood, MSZoning, Foundation | One-Hot o Feature Hashing |
| **Ordinal** | Categorias con orden conocido | ExterQual (Po<Fa<TA<Gd<Ex), KitchenQual | Codificacion ordinal (entero) |

**¿Por que importa?**

- **Nominal con codificacion ordinal** → error: introduces un orden falso que el modelo
  interpreta como distancia (p.ej. "RL" < "RM" no tiene sentido).
- **Ordinal con one-hot** → desperdicio: pierdes la informacion del orden que correlaciona
  con el precio (una cocina `Ex` vale mas que una `TA`).

**Escala de calidad en Ames:**

$$\text{None}(0) < \text{Po}(1) < \text{Fa}(2) < \text{TA}(3) < \text{Gd}(4) < \text{Ex}(5)$$

Aqui `None=0` significa que la feature no existe en la casa (sin piscina, sin chimenea).

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

kitchen_raw = np.array(["TA", "Gd", "Ex", "TA", "Fa", "Gd", "TA", "Ex"]).reshape(-1, 1)
nbhd_raw    = np.array(["NAmes", "CollgCr", "OldTown", "NAmes", "Edwards",
                        "CollgCr", "Gilbert", "NAmes"]).reshape(-1, 1)

QUALITY_ORDER = [["None", "Po", "Fa", "TA", "Gd", "Ex"]]
oe = OrdinalEncoder(categories=QUALITY_ORDER)
kitchen_ord = oe.fit_transform(kitchen_raw).astype(int)

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
nbhd_oh = ohe.fit_transform(nbhd_raw)

print("=== ORDINAL — KitchenQual (orden tiene significado) ===")
for raw, cod in zip(kitchen_raw.ravel(), kitchen_ord.ravel()):
    print(f"  {raw} -> {cod}")

print("\n=== ONE-HOT — Neighborhood (sin orden natural) ===")
print("Categorias:", list(ohe.categories_[0]))
for raw, vec in zip(nbhd_raw.ravel()[:4], nbhd_oh[:4]):
    print(f"  {raw:10s} -> {vec.astype(int)}")

print("\n-> Ordinal preserva el orden significativo (Ex=5 > TA=3)")
print("-> One-hot no impone orden (todos los barrios son equidistantes)")

## 8. Fuga de datos (data leakage) — la regla de oro <a id="c8"></a>

**Definicion.** Fuga de datos ocurre cuando **informacion que no estaria disponible en
el momento de prediccion** se cuela en el entrenamiento. Sintoma: metricas de validacion
excelentes que se derrumban en produccion.

### Dos tipos

1. **Target leakage:** una feature contiene informacion del objetivo o del futuro.
   - Ejemplo: `price_per_sqft` calculado *despues* de conocer `SalePrice`.

2. **Contaminacion train/test:** ajustar (`fit`) los transformadores sobre TODO el
   dataset antes de separar train/test.
   - El scaler/imputer/encoder ve estadisticas del test → evaluacion inflada.

### La regla de oro

> **Separa primero, ajusta solo en train.**
> - `fit` → SOLO en `X_train`
> - `transform` → en `X_train` Y en `X_test` (con los parametros aprendidos en train)
> - **Nunca** `fit` con el test.

### Paradigma fit / transform / fit_transform

| Metodo | Que hace | Cuando usar |
|---|---|---|
| `fit(X)` | Aprende parametros de X (media, mediana, categorias...) | Solo en train |
| `transform(X)` | Aplica los parametros ya aprendidos | En train y en test |
| `fit_transform(X)` | fit + transform en un paso | Solo en train |

In [ ]:
X_full = rng.lognormal(7.3, 0.3, (500, 1))
X_train_d, X_test_d = train_test_split(X_full, test_size=0.3, random_state=0)

scaler_leak = StandardScaler().fit(X_full)
media_con_fuga = scaler_leak.mean_[0]

scaler_ok = StandardScaler().fit(X_train_d)
media_sin_fuga = scaler_ok.mean_[0]

X_test_malo = scaler_leak.transform(X_test_d)
X_test_bien = scaler_ok.transform(X_test_d)

print("Comparacion del valor aprendido por el scaler:")
print(f"  fit en TODO el dataset (con fuga):  media = {media_con_fuga:.4f}")
print(f"  fit solo en TRAIN (correcto):       media = {media_sin_fuga:.4f}")
print(f"  Diferencia: {abs(media_con_fuga - media_sin_fuga):.6f}")
print()
print("Diferencia en la transformacion del test:")
dif = np.abs(X_test_malo - X_test_bien)
print(f"  MAE entre las dos versiones del test escalado: {dif.mean():.6f}")
print()
print("-> Con pocos datos o mucho sesgo, la diferencia puede ser GRANDE")
print("-> La solucion limpia: usar Pipeline (ver concepto 9)")

## 9. Pipeline y el paradigma fit / transform <a id="c9"></a>

**`Pipeline` de sklearn** encadena una secuencia de transformadores + un estimador final.

- `pipe.fit(X_train, y_train)` → internamente hace `fit_transform` en cada paso intermedio
  y `fit` en el ultimo (el modelo).
- `pipe.predict(X_test)` → hace solo `transform` en cada paso intermedio.

**Ventaja clave:** dentro de `cross_val_score` o `GridSearchCV`, el pipeline se
**re-ajusta solo con el fold de entrenamiento** en cada iteracion. Nunca ve el fold
de validacion durante el ajuste → **sin fuga en validacion cruzada**.

**`ColumnTransformer`** aplica transformadores distintos a columnas distintas:
- Columnas numericas → `SimpleImputer` + `StandardScaler`
- Columnas categoricas → `SimpleImputer` + `OneHotEncoder`

```
Pipeline:
  ColumnTransformer
    ├── numericas: SimpleImputer(mediana) → StandardScaler
    └── categoricas: SimpleImputer(moda) → OneHotEncoder
  └── Modelo (Ridge, RandomForest, ...)
```

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

rng2 = np.random.default_rng(1)
n = 300
X_demo = pd.DataFrame({
    "area":    rng2.normal(1500, 300, n),
    "calidad": rng2.integers(3, 10, n).astype(float),
    "zona":    rng2.choice(["RL", "RM", "FV"], n),
})
X_demo.loc[rng2.random(n) < 0.1, "area"] = np.nan
y_demo = 50 * X_demo["area"].fillna(1500) + 10000 * X_demo["calidad"] + rng2.normal(0, 20000, n)

num_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc",  StandardScaler()),
])
cat_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
pre = ColumnTransformer([("num", num_pipe, ["area", "calidad"]),
                          ("cat", cat_pipe, ["zona"])])
pipe = Pipeline([("pre", pre), ("ridge", Ridge(alpha=1.0))])

scores = cross_val_score(pipe, X_demo, y_demo, cv=5,
                          scoring="neg_root_mean_squared_error")
print("RMSE por fold (Pipeline con 5-fold CV, sin fuga):")
print(" ", (-scores).round(0))
print(f"RMSE medio: {(-scores).mean():.0f}  +/- {(-scores).std():.0f}")
print()
print("-> El pipeline garantiza que el imputer y el scaler solo")
print("   ven el fold de entrenamiento en cada iteracion de CV.")

## 10. PCA — varianza explicada y componentes principales <a id="c10"></a>

**PCA (Analisis de Componentes Principales)** es un metodo de **extraccion** de variables
(crea nuevas columnas combinando las originales, a diferencia de la *seleccion* que elige
columnas existentes).

**Idea:** encontrar las direcciones ortogonales (componentes) que maximizan la varianza.

**Formula:** se descompone la matriz de covarianza $\Sigma = \frac{1}{n-1}X^\top X$ en
autovectores: $\Sigma v_k = \lambda_k v_k$.

**Varianza explicada** por la componente $k$:
$$\text{EVR}_k = \frac{\lambda_k}{\sum_j \lambda_j}$$

**Reglas practicas:**
- **Siempre estandariza antes** de PCA. Sin estandarizar, las features de mayor varianza
  (en sus unidades originales) dominan todas las componentes.
- Elige el numero de componentes segun la varianza acumulada (p.ej. retener el 95%).
- PCA **pierde interpretabilidad**: las componentes no tienen un nombre de dominio.

| Componente | Varianza capturada | Varianza acumulada |
|---|---|---|
| PC1 | Mayor | Mayor |
| PC2 | Segunda mayor | Mayor + segunda |
| ... | ... | ... |

In [ ]:
rng3 = np.random.default_rng(7)
n_pts = 250
t = rng3.uniform(0, 2 * np.pi, n_pts)
X_pca = np.column_stack([
    3 * np.cos(t) + rng3.normal(0, 0.5, n_pts),
    np.sin(t) + rng3.normal(0, 0.2, n_pts),
])
X_pca_std = StandardScaler().fit_transform(X_pca)

pca = PCA().fit(X_pca_std)
evr = pca.explained_variance_ratio_
Z = pca.transform(X_pca_std)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].scatter(X_pca_std[:, 0], X_pca_std[:, 1], s=15, alpha=0.5, color="#4c78a8")
for i, v in enumerate(pca.components_):
    scale = np.sqrt(pca.explained_variance_[i])
    axes[0].annotate("", xy=v * scale, xytext=(0, 0),
                     arrowprops=dict(arrowstyle="->", color=["#e45756", "#54a24b"][i], lw=2))
axes[0].set_title("Datos originales y componentes")
axes[0].set_xlabel("X1"); axes[0].set_ylabel("X2")

axes[1].bar(["PC1", "PC2"], evr, color=["#e45756", "#54a24b"], alpha=0.8)
axes[1].plot(["PC1", "PC2"], np.cumsum(evr), "o-k", label="acumulada")
axes[1].set_ylim(0, 1.1); axes[1].set_ylabel("fraccion de varianza")
axes[1].set_title(f"Scree plot\nPC1={evr[0]:.1%}  PC2={evr[1]:.1%}")
axes[1].legend()

axes[2].scatter(Z[:, 0], Z[:, 1], s=15, alpha=0.5, color="#f58518")
axes[2].set_title("Proyeccion en PC1/PC2")
axes[2].set_xlabel("PC1"); axes[2].set_ylabel("PC2")

plt.suptitle("PCA: las componentes son las direcciones de maxima varianza",
             fontweight="bold")
plt.tight_layout(); plt.show()
print(f"Varianza explicada: PC1={evr[0]:.3f}  PC2={evr[1]:.3f}  Total={evr.sum():.3f}")

## 11. Metricas de regresion: RMSE, MAE, R2, RMSLE <a id="c11"></a>

| Metrica | Formula | Unidades | Sensible a outliers | Interpretacion |
|---|---|---|---|---|
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | mismas que $y$ | Si (cuadratica) | Error tipico |
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | mismas que $y$ | No (lineal) | Error mediano |
| **R2** | $1 - \frac{\sum(y_i-\hat{y}_i)^2}{\sum(y_i-\bar{y})^2}$ | sin unidades $[0,1]$ | Si | Fraccion de varianza explicada |
| **RMSLE** | $\sqrt{\frac{1}{n}\sum(\log(1+y_i) - \log(1+\hat{y}_i))^2}$ | log-escala | Poco | Errores relativos |

**RMSLE (Root Mean Squared Log Error)** es la metrica oficial de la competencia Kaggle de
Ames Housing. Al trabajar en escala logaritmica, un error del mismo porcentaje en una
casa barata y en una cara **pesa igual**. Equivale a entrenar el modelo sobre
`log1p(SalePrice)` y evaluar con RMSE.

**Regla practica:**
- RMSE: cuando los errores grandes son muy costosos.
- MAE: cuando quieres robustez ante valores atipicos.
- R2: para comunicar el poder explicativo del modelo.
- RMSLE: cuando el error importa en terminos **relativos** (porcentaje).

In [ ]:
y_real = np.array([180000, 220000, 140000, 300000, 500000, 95000, 260000, 195000])
y_pred = np.array([190000, 210000, 155000, 280000, 420000, 110000, 265000, 200000])

rmse  = np.sqrt(mean_squared_error(y_real, y_pred))
mae   = mean_absolute_error(y_real, y_pred)
r2    = r2_score(y_real, y_pred)
rmsle = np.sqrt(mean_squared_error(np.log1p(y_real), np.log1p(y_pred)))

errores = y_pred - y_real
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

idx = np.arange(len(y_real))
axes[0].bar(idx, errores / 1000, color=["#e45756" if e > 0 else "#4c78a8" for e in errores],
            alpha=0.8, edgecolor="white")
axes[0].axhline(0, color="black", lw=1)
axes[0].set_title("Errores de prediccion (pred - real) en miles USD")
axes[0].set_xlabel("casa"); axes[0].set_ylabel("error (k USD)")

axes[1].scatter(y_real / 1000, y_pred / 1000, color="#4c78a8", s=60, edgecolor="white")
lims = [min(y_real.min(), y_pred.min()) / 1000, max(y_real.max(), y_pred.max()) / 1000]
axes[1].plot(lims, lims, "--", color="#e45756", lw=1.5, label="prediccion perfecta")
axes[1].set_xlabel("SalePrice real (k USD)"); axes[1].set_ylabel("SalePrice predicho (k USD)")
axes[1].set_title(f"Predicho vs Real  R2={r2:.3f}")
axes[1].legend()

plt.suptitle("Metricas de regresion en Ames Housing", fontweight="bold")
plt.tight_layout(); plt.show()

print(f"RMSE  = ${rmse:,.0f}  (error cuadratico medio, penaliza mas los errores grandes)")
print(f"MAE   = ${mae:,.0f}  (error absoluto medio, robusto a outliers)")
print(f"R2    = {r2:.4f}     (fraccion de varianza explicada; 1.0 = perfecto)")
print(f"RMSLE = {rmsle:.4f}  (metrica oficial Kaggle; error en escala log)")

## 12. Feature Store y sesgo entrenamiento-serving <a id="c12"></a>

### Sesgo entrenamiento-serving (Training-Serving Skew)

Ocurre cuando las features que el modelo recibe **en produccion** se calculan de forma
diferente a como se calcularon durante el entrenamiento. Consecuencia: el modelo opera
fuera de distribucion y se degrada silenciosamente.

**Causas comunes:**
- La mediana de imputacion se recalcula con datos nuevos en produccion.
- El escalado usa parametros distintos en serving vs entrenamiento.
- Distintos equipos reimplementan la misma feature de forma diferente.

### Feature Store — la solucion

Un **feature store** es una plataforma que:
1. Define las features **una vez** (codigo, definicion, TTL).
2. Sirve las mismas features tanto para entrenamiento como para inferencia online.
3. Garantiza **correctitud point-in-time**: al construir datasets de entrenamiento,
   solo usa informacion disponible antes del evento (sin mirar el futuro).

```
Datos crudos  ->  Feature Engineering  ->  OFFLINE store (historia, parquet)
                                                  |
                                          feast materialize
                                                  v
                                           ONLINE store (Redis, ultimo valor)

Entrenamiento  <-- get_historical_features  (mismo codigo de features)
Serving        <-- get_online_features      (mismo codigo de features)
```

| Concepto | Descripcion |
|---|---|
| **Offline store** | Historia completa con timestamps (parquet). Para `get_historical_features`. |
| **Online store** | Ultimo valor por entidad (Redis). Para `get_online_features` con baja latencia. |
| **Point-in-time** | Al armar el dataset, cada fila usa solo features disponibles antes de `event_timestamp`. |
| **Materializacion** | `feast materialize` empuja del offline al online store. |

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(13, 6))
ax.set_xlim(0, 13); ax.set_ylim(0, 7); ax.axis("off")

def caja(x, y, w, h, texto, color, fontsize=9):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.07",
                                fc=color, ec="#444", lw=1.3))
    ax.text(x + w/2, y + h/2, texto, ha="center", va="center",
            fontsize=fontsize, wrap=True)

def flecha(p1, p2, label="", color="#333"):
    ax.add_patch(FancyArrowPatch(p1, p2, arrowstyle="-|>",
                                 mutation_scale=16, color=color, lw=1.5))
    if label:
        mx = (p1[0]+p2[0])/2; my = (p1[1]+p2[1])/2
        ax.text(mx, my+0.2, label, ha="center", fontsize=8,
                color=color, style="italic")

caja(0.2, 4.5, 2.2, 1.2, "Datos\ncrudos", "#cfe8ff")
caja(3.2, 4.5, 3.0, 1.2, "OFFLINE store\n(parquet, historia)", "#d6f5d6")
caja(8.0, 4.5, 3.5, 1.2, "ONLINE store\n(Redis, ultimo valor)", "#ffe2b3")
caja(3.2, 2.5, 3.0, 1.0, "REGISTRY\n(Feast)", "#f3d1ff")
caja(0.2, 0.5, 4.5, 1.2, "get_historical_features\n(entrenamiento, point-in-time)", "#e8e8e8")
caja(7.8, 0.5, 4.5, 1.2, "get_online_features\n(serving, ms latencia)", "#e8e8e8")

flecha((2.4, 5.1), (3.2, 5.1), "FE")
flecha((6.2, 5.1), (8.0, 5.1), "materialize", "#54a24b")
flecha((4.7, 4.5), (4.7, 3.5))
flecha((4.7, 2.5), (2.5, 1.7), "mismas\nfeatures", "#1f77b4")
flecha((9.5, 4.5), (9.8, 1.7), "mismas\nfeatures", "#d62728")

ax.text(6.5, 6.5, "Feature Store (Feast): elimina el sesgo entrenamiento-serving",
        ha="center", fontsize=12, fontweight="bold")
ax.text(6.5, 6.0,
        "Las MISMAS definiciones de features sirven tanto para entrenar como para predecir",
        ha="center", fontsize=9, color="gray")

plt.tight_layout(); plt.show()
print("Clave: el contrato que elimina el sesgo es que AMBOS caminos")
print("(entrenamiento y serving) usan el mismo codigo de features.")